# AIMS SENEGAL JAN 2026
## Aurelle TCHAGNA, PhD
# Numerical Optimization Lab : Smart Input Allocation under Budget & Environmental Constraints

**Context (Africa-relevant):**  
A farmers' cooperative wants to increase crop yield and profit across many plots, but has **limited fertilizer and irrigation water**. Over-application of fertilizer increases **nitrate leaching** and environmental risk.

You are given a dataset of plot-level observations (soil, rainfall, pests,  NDVI (Normalized Difference Vegetation Index) time series tracking plant health , yield).  
You will:
1. **Preprocess** the data (missing values, outliers, scaling).
2. Use **numerical differentiation** on NDVI to create new features.
3. Fit a simple **yield response model** with diminishing returns.
4. Formulate a **convex optimization** problem to allocate *additional* fertilizer and irrigation across plots.
5. Write the **Lagrangian** and interpret **KKT conditions**.
6. Solve the optimization problem using a Python optimization library.
7. Verify constraints, interpret results, and visualize the allocation.

---

## Files
- `agri_numerical_optimization_lab_data.csv`



In [1]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional (recommended) convex optimization library
try:
    import cvxpy as cp
    CVXPY_OK = True
except Exception as e:
    CVXPY_OK = False
    print("cvxpy not available:", e)

np.random.seed(0)


In [11]:

#Import your data
# display the head of the data



## Part 1 — Data quality checks (Preprocessing)

### Questions
1. Check missing values per column. Which variables have missing data?
2. Detect obvious outliers (e.g., extremely high rainfall, fertilizer, pest index) and decide a cleaning rule.
3. Create a clean dataframe `df_clean`.

### Notes
- In real deployments, you would consult agronomists for realistic ranges.
- Here, you can use robust rules like **IQR clipping** or **percentile clipping**.



## Part 2 — Numerical differentiation as feature engineering (NDVI)

NDVI (Normalized Difference Vegetation Index) is a remote-sensing proxy for plant greenness/vigor.

We have NDVI measurements for 6 weeks: `ndvi_w1 ... ndvi_w6`.

### Questions
1. Compute a **forward-difference slope** at week 1:  
   $$ s_1 \approx (NDVI_2 - NDVI_1)/h $$, with $h=1$ week.
2. Compute a **central-difference slope** at weeks 2–5:  
   $$ s_k \approx (NDVI_{k+1}-NDVI_{k-1})/(2h) $$.
3. Compute an average NDVI slope `ndvi_slope_mean` and an average second derivative `ndvi_accel_mean`.
4. Explain why slope/acceleration can help predict yield.



NDVI slope and acceleration encode how fast and how sustainably a crop is growing, which is often more predictive of final yield than NDVI values alone.
NDVI (Normalized Difference Vegetation Index) is a proxy for plant health and photosynthetic activity.

Low NDVI → sparse or stressed vegetation

High NDVI → dense, healthy canopy

Yield (biomass / grain) depends on how long and how strongly plants photosynthesize


## Part 3 — Fit a yield-response model (diminishing returns)

We want a model that captures **diminishing returns** of fertilizer and irrigation.

A simple and useful choice is a **concave quadratic** (in the input variables):
$$
\hat y = c + a_1 f + a_2 i + a_3 r + a_4 N + a_5 s - b_1 f^2 - b_2 i^2
$$
where:
- $f$ = fertilizer (kg/ha)
- $i$ = irrigation (mm equiv)
- $r$ = rainfall
- $N$ = soil nitrogen
- $s$ = NDVI slope feature (proxy for vigor)

### Questions
1. Build the design matrix including $f, i, r, N, s, f^2, i^2$.
2. Fit the parameters by least squares.




## Part 4 — Optimization problem (allocate *additional* inputs)

Now the cooperative has budgets:
- Total extra fertilizer available: $B_f$(kg/ha aggregated across plots, scaled by area)
- Total extra irrigation available: $B_i$ (mm equiv aggregated across plots, scaled by area)

Decision variables per plot \(k\):
- $\Delta f_k \ge 0$: additional fertilizer to apply (kg/ha)
- $\Delta i_k \ge 0$: additional irrigation (mm equiv)

We use the fitted model to predict yield after adding inputs:
$$
\hat y_k(\Delta f_k, \Delta i_k) =
c_k + a_1 (f_k+\Delta f_k) + a_2 (i_k+\Delta i_k)
- b_1 (f_k+\Delta f_k)^2 - b_2 (i_k+\Delta i_k)^2
$$
(Other terms are constants per plot.)

Assume:
- price per ton: $p$ USD/ton
- fertilizer cost: $c_f$cUSD per (kg/ha)
- irrigation cost: $c_i$ USD per (mm equiv)

**Objective:** maximize total profit across plots (area-weighted):
$$
\max \sum_k \text{area}_k \Big(p\,\hat y_k - c_f\,(f_k+\Delta f_k) - c_i\,(i_k+\Delta i_k)\Big)
$$

**Environmental constraint (nitrate leaching proxy):**
$$
\sum_k \text{area}_k \,(f_k+\Delta f_k) \le L_{max}
$$
and budgets:
$$
\sum_k \text{area}_k \,\Delta f_k \le B_f, \quad
\sum_k \text{area}_k \,\Delta i_k \le B_i
$$

### Questions
1. Show that (with $b_1>0, b_2>0$) the objective is **concave** in $\Delta f, \Delta i$.  
2. Write the **Lagrangian** and the **KKT conditions** (stationarity, primal feasibility, dual feasibility, complementary slackness).
3. Solve the problem numerically (QP) and interpret results.




## Part 5  KKT interpretation 

For a maximization of a concave function with convex constraints, KKT conditions are **necessary and sufficient**.

- If a budget constraint is **tight**, its dual variable is positive and represents a **shadow price** (marginal value of increasing the budget by 1 unit).
- Complementary slackness: if a constraint is not tight (has slack), its multiplier should be ~0.

### Quick check
- If fertilizer budget used is equal to $B_f$, you should see $\lambda_{B_f} > 0$.
- If not, you expect $\lambda_{B_f} \approx 0$.




## Exercises 

1. **Sensitivity:** Increase $B_f$ by 20% and re-solve. Compare the new profit and the dual variable $\lambda_{B_f}$.  
2. **Environmental policy:** Decrease $L_{max}$ by 10% and observe how allocations change.  
3. **Convexity check:** Numerically verify concavity of the per-plot profit in $\Delta f$ by plotting it for a single plot.  
4. **Equity constraint (bonus):** Add a constraint that each region must receive at least 25% of the irrigation budget.

